# 03 · Cross-validation

Two methods:
- **k-fold**: every row is scored once, by a model that never saw it.
- **repeated holdout**: `N_SPLITS` independent random test sets of `TEST_SIZE`. Rows can
  appear in several or none, but each confidence bucket gets one accuracy per split.

**Runs can span several sessions.** Each split writes its predictions to
`outputs/cv_<stamp>/split_<k>/` as soon as it finishes. To continue, set
`RESUME_RUN` to the run's folder name and rerun the notebook: finished splits are
skipped, and the run's saved model config and data settings are used, so nothing
can drift between sessions. `SPLITS_TO_RUN` limits a session to particular splits.
A split interrupted part way through is simply rerun.

For the NN, each split carves a validation slice from its training rows for early
stopping and checkpointing; the split's test rows are never used for either.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # repo root, so `import classifier` works

DATA_DIR, CONFIG_DIR, OUTPUT_DIR = "../data", "../config", "../outputs"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from classifier import io, evaluation, plotting
from classifier.features import VectorizerSettings
from classifier.nb import NBSettings
from classifier.model import CategoryClassifier, ModelConfig, validation_split, hierarchy_lookup

from classifier import crossval

## Parameters

In [ ]:
# ---- run control ---------------------------------------------------------------------
RESUME_RUN = None                  # e.g. "cv_20260930_150000" to continue; None = new run
SPLITS_TO_RUN = None               # None = every unfinished split; e.g. [0, 1] for this session
ANALYSE_PARTIAL = False            # analyse before every split has finished

# ---- a new run (ignored when resuming: the run's saved settings apply) -----------------
CLEANING_STAMP = "YYYYMMDD_HHMMSS"
BLACKLIST_FILE = None              # in data/: IDs removed from everything
EXCLUDE_FILE = "to_predict.txt"    # in data/: IDs kept out of CV, e.g. the test set. None = off
METHOD = "kfold"                   # "kfold" | "holdout"
N_SPLITS = 5
TEST_SIZE = 0.2                    # holdout only
STRATIFY = True                    # on the leaf category
SEED = 42
NN_VAL_FRAC = 0.05                 # NN only: validation slice of each split's training rows
SAVE_SPLIT_MODELS = False          # keep each split's fitted model (large for the NN)

# ---- which model --------------------------------------------------------------
MODEL_TYPE = "nb"                  # "nb" (benchmark) | "nn"

# ---- rare-token pruning, counted on training rows only --------------------------
MIN_TOKEN_COUNT = 1                # 1 = off; 5 keeps tokens seen in >= 5 training descriptions

# ---- vectorizer (both models) ---------------------------------------------------
VECTORIZER = VectorizerSettings(
    kind="count",                  # "count" | "tfidf"
    ngram_range=(1, 2),
    unordered_bigrams=True,        # bidirectional bigrams: 'ball valve' == 'valve ball'
    min_df=3,                      # int = document count
    max_df=1.0,                    # float = proportion
    max_features=None,             # None = no cap
    stop_words=None,               # "english" also drops FRONT, BACK, BOTTOM, FILL ...
    binary=True,
    # tfidf only
    sublinear_tf=True, norm="l2", use_idf=True, smooth_idf=True,
)

# ---- Naive Bayes -----------------------------------------------------------------
NB = NBSettings(alpha=0.001, fit_prior=True, class_prior=None)

# ---- neural network (ignored for NB) --------------------------------------------
NN = dict(
    hidden_dims=(512, 256), dropout=0.2, batch_norm=True, feature_dropout=0.0,
    batch_size=512, epochs=100, lr=1e-3, weight_decay=1e-5, label_smoothing=0.0,
    level_loss_weights=None,       # e.g. (1, 1, 1, 2) to weight the deepest level
    sparse_grad=True,              # SparseAdam on the first layer; set False if torch objects
    lr_scheduler=True, scheduler_patience=3, scheduler_factor=0.5,
    early_stopping=True, patience=6,
    monitor="accuracy",            # "accuracy" (deepest level) | "val_loss"
    monitor_level=-1,
    checkpoint=True,               # best epoch kept on disk and restored
    prefer_gpu=True,               # CUDA if available, else CPU
    num_workers=0, eval_batch_size=4096, seed=42,
)

# ---- turning per-level probabilities into paths ----------------------------------
PATH_MODE = "independent"          # "independent" | "conditional"
NORMALISE = True                   # normalise path scores over valid paths
SCORE_BATCH_SIZE = 2000            # rows per path-scoring block (memory)
PREDICT_CHUNK_SIZE = 50000         # rows per prediction chunk (memory)

# ---- analysis ---------------------------------------------------------------------------
BUCKET_WIDTH = 0.05
N_QUANTILE_BUCKETS = None
REFERENCE_ACCURACY = 0.95          # dashed reference line on the charts; None = off
TOP_CONFUSIONS = 20
NOTES = ""

## 1 · Start or resume the run

In [ ]:
if RESUME_RUN:
    RUN_DIR = f"{OUTPUT_DIR}/{RESUME_RUN}"
    STAMP = RESUME_RUN.replace("cv_", "")
    settings = io.load_json(f"{RUN_DIR}/settings.json")
    CLEANING_STAMP = settings["cleaning_stamp"]
    BLACKLIST_FILE, EXCLUDE_FILE = settings["blacklist_file"], settings["exclude_file"]
    print(f"resuming {RUN_DIR}: {settings['method']}, {settings['n_splits']} splits")
else:
    STAMP = io.stamp()
    RUN_DIR = f"{OUTPUT_DIR}/cv_{STAMP}"

cleaning_config, cleaner = io.load_cleaning(CONFIG_DIR, CLEANING_STAMP)
df = io.load_labelled(DATA_DIR, cleaning_config, ID_COL,
                      blacklist_file=BLACKLIST_FILE, exclude_file=EXCLUDE_FILE)
lookup = hierarchy_lookup(df, LEVEL_COLUMNS, NAME_COLUMNS)

if not RESUME_RUN:
    cfg = ModelConfig(
        model_type=MODEL_TYPE, text_col=TEXT_COL, id_col=ID_COL,
        level_columns=LEVEL_COLUMNS, name_columns=NAME_COLUMNS,
        min_token_count=MIN_TOKEN_COUNT, vectorizer=VECTORIZER, nb=NB, nn=dict(NN),
        path_mode=PATH_MODE, normalise=NORMALISE, score_batch_size=SCORE_BATCH_SIZE,
        predict_chunk_size=PREDICT_CHUNK_SIZE,
        cleaning_stamp=CLEANING_STAMP, blacklist_file=BLACKLIST_FILE, notes=NOTES,
    )
    cfg.source = f"cv_{STAMP}"
    crossval.start_run(RUN_DIR, df, cfg, METHOD, N_SPLITS, test_size=TEST_SIZE, stratify=STRATIFY,
                       seed=SEED, nn_val_frac=NN_VAL_FRAC, save_split_models=SAVE_SPLIT_MODELS,
                       extra={"cleaning_stamp": CLEANING_STAMP, "blacklist_file": BLACKLIST_FILE,
                              "exclude_file": EXCLUDE_FILE})
    cfg.save(f"{CONFIG_DIR}/model_{STAMP}.json")
    print(f"\nRESUME_RUN = \"cv_{STAMP}\"                <- to continue this run later")
    print(f"MODEL_CONFIG_FILE = \"model_{STAMP}.json\"   <- to train this config in notebook 04")

settings, cfg, assignment = crossval.load_run(RUN_DIR, df)     # checks the data matches the run
print(f"model: {cfg.model_type}, path mode: {cfg.path_mode}")
crossval.status(RUN_DIR)

## 2 · Run splits

The expensive cell. Safe to stop between splits.

In [ ]:
crossval.run_splits(RUN_DIR, df, splits=SPLITS_TO_RUN)

## 3 · Results

In [ ]:
st = crossval.status(RUN_DIR)
if not st["done"].all() and not ANALYSE_PARTIAL:
    raise SystemExit(f"{int((~st['done']).sum())} split(s) still to run; "
                     "rerun with RESUME_RUN set, or set ANALYSE_PARTIAL = True")
pred = crossval.collect(RUN_DIR, df, allow_partial=ANALYSE_PARTIAL)
SPLIT_COL = "split"
print(f"{len(pred):,} out-of-sample predictions over {pred['split'].nunique()} split(s)")

In [ ]:
# Per split, then mean and std across splits
by_split = evaluation.metrics_by_split(pred, LEVEL_COLUMNS)
by_split

### Overall

In [ ]:
m = evaluation.metrics(pred, LEVEL_COLUMNS)
display(pd.Series(m, name="value").to_frame())
fig = plotting.plot_per_level_accuracy(m, LEVEL_COLUMNS)

### By confidence bucket

`L<d> Confidence` is the share of path probability under the predicted prefix down to
depth d. Each depth is bucketed on its own distribution; read each panel on its own.

In [ ]:
buckets = evaluation.bucket_accuracy(pred, LEVEL_COLUMNS, split_col=SPLIT_COL,
                                     bucket_width=BUCKET_WIDTH, n_quantile_buckets=N_QUANTILE_BUCKETS)
summaries = evaluation.bucket_summary(buckets)
summaries[len(LEVEL_COLUMNS)]

In [ ]:
figs = {
    "accuracy_by_bucket": plotting.plot_accuracy_by_bucket(summaries[len(LEVEL_COLUMNS)],
                                                           target_accuracy=REFERENCE_ACCURACY),
    "coverage": plotting.plot_coverage_curve(summaries[len(LEVEL_COLUMNS)], REFERENCE_ACCURACY),
    "accuracy_by_bucket_per_level": plotting.plot_accuracy_by_bucket_grid(
        summaries, LEVEL_COLUMNS, target_accuracy=REFERENCE_ACCURACY),
    "coverage_per_level": plotting.plot_coverage_curve_grid(
        summaries, LEVEL_COLUMNS, target_accuracy=REFERENCE_ACCURACY),
}

### Spread across splits

One point per split in each bucket; violins where a bucket has three or more.

In [ ]:
figs["violins_per_level"] = plotting.plot_bucket_violins_by_level(buckets, LEVEL_COLUMNS)
figs["violins"] = plotting.plot_bucket_violins(buckets, len(LEVEL_COLUMNS))

### Most frequent confusions

In [ ]:
confusions = evaluation.top_confusions(pred, LEVEL_COLUMNS[-1], n=TOP_CONFUSIONS,
                                       lookup=lookup, name_col=NAME_COLUMNS[-1])
figs["confusions"] = plotting.plot_top_confusions(confusions, LEVEL_COLUMNS[-1])
confusions

In [ ]:
unreadable = pred[~pred["readable"].astype(bool)]
print(f"{len(unreadable):,} of {len(pred):,} rows had no feature the model knows "
      f"(their predictions come from class frequencies alone)")
unreadable[[ID_COL, TEXT_COL]].head(10)

### NN training, per split

In [ ]:
if cfg.model_type == "nn":
    for k in st.loc[st["done"], "split"]:
        path = f"{RUN_DIR}/split_{k}/history.csv"
        if os.path.exists(path):
            figs[f"training_curves_split_{k}"] = plotting.plot_training_curves(pd.read_csv(path), LEVEL_COLUMNS)
    print("best epoch per split:", st.set_index("split")["best_epoch"].to_dict())
    print("median best epoch:", crossval.median_best_epoch(RUN_DIR),
          "(notebook 04 can train for this many epochs without a validation slice)")

## 4 · Save

In [ ]:
analysis_dir = f"{RUN_DIR}/analysis"
os.makedirs(analysis_dir, exist_ok=True)
pred.to_csv(f"{analysis_dir}/out_of_sample_predictions.csv", index=False)
by_split.to_csv(f"{analysis_dir}/metrics_by_split.csv", index=False)
buckets.drop(columns="bucket").to_csv(f"{analysis_dir}/bucket_accuracy.csv", index=False)
io.save_json({"metrics": m, "complete": bool(st["done"].all()),
              "median_best_epoch": crossval.median_best_epoch(RUN_DIR) if cfg.model_type == "nn" else None},
             f"{analysis_dir}/summary.json")
for name, f in figs.items():
    f.savefig(f"{analysis_dir}/{name}.png", dpi=120, bbox_inches="tight")
print("saved to", analysis_dir)